# Multi-Agent Hospital Environment for MAPPO
Reusable hospital emergency department environment only; no MAPPO, training loop, neural network, grid-world, or movement physics is implemented.


## Metric Notes
Environment metrics include delta_queue, delta_capacity, waiting time, queue pressure, and release/arrival probabilities. MAPPO cells use eps_clip and GAE delta only when training is enabled.


## Imports
Core imports cover queues, dataclasses, typing, NumPy randomness, and optional Gymnasium spaces.


In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from typing import Any, Deque, Dict, Iterable, List, Optional, Tuple

import numpy as np

try:
    from gymnasium import spaces
except ImportError:  # pragma: no cover - fallback for lightweight use.
    spaces = None


## Static Configuration
Staff schedules, patient stage labels, patient types, and agent action meanings are fixed here for reproducibility.


In [ ]:
STAFF_SCHEDULE = {
    "day": {
        "nurses": 6,
        "doctors": 5,
        "physicians": 3,
        "radiologists": 2,
        "receptionists": 3,
        "administrators": 2,
        "paramedics": 4,
    },
    "evening": {
        "nurses": 4,
        "doctors": 3,
        "physicians": 2,
        "radiologists": 1,
        "receptionists": 2,
        "administrators": 1,
        "paramedics": 3,
    },
    "night": {
        "nurses": 2,
        "doctors": 1,
        "physicians": 1,
        "radiologists": 1,
        "receptionists": 1,
        "administrators": 1,
        "paramedics": 2,
    },
}


PATIENT_TYPES = {
    0: "emergency",
    1: "urgent",
    2: "elective_minor",
}


PATIENT_STAGES = {
    0: "arrival",
    1: "registration",
    2: "assessment",
    3: "triage",
    4: "imaging_tests",
    5: "treatment",
    6: "resus",
    7: "major",
    8: "minor",
    9: "general_admission",
    10: "icu_admission",
    11: "discharge",
    12: "delay_wait",
}


ACTION_MEANINGS = {
    "reception_agent": {
        0: "do_nothing",
        1: "register_patient",
        2: "move_to_assessment",
    },
    "nurse_triage_agent": {
        0: "do_nothing",
        1: "send_to_minor",
        2: "send_to_major",
        3: "send_to_resus",
        4: "delay_assessment",
    },
    "doctor_agent": {
        0: "do_nothing",
        1: "treat",
        2: "request_imaging",
        3: "escalate",
        4: "deescalate",
        5: "delay_treatment",
    },
    "physician_agent": {
        0: "do_nothing",
        1: "admit_general",
        2: "admit_icu",
        3: "discharge",
        4: "delay_admission",
    },
    "radiologist_agent": {
        0: "do_nothing",
        1: "process_imaging",
        2: "delay_imaging",
    },
    "ambulance_paramedic_agent": {
        0: "do_nothing",
        1: "bring_emergency_patient",
        2: "redirect_noncritical_patient",
    },
    "administrator_agent": {
        0: "do_nothing",
        1: "prioritise_emergency_flow",
        2: "prioritise_general_throughput",
        3: "preserve_icu_capacity",
        4: "increase_discharge_pressure",
    },
}


## Patient Record
Each patient stores type, stage, waiting time, imaging need, and current area. delta_stage is driven by agent decisions, not physical movement.


In [ ]:
@dataclass
class Patient:
    """Lightweight operational patient record, not a spatial entity."""

    patient_type: int
    stage: int = 0
    waiting_time: int = 0
    needs_imaging: bool = False
    area: Optional[str] = None


## Space Fallbacks
Gymnasium spaces are used when installed; fallback spaces keep the notebook runnable. These define action/observation shapes, not policies.


In [ ]:
class _Discrete:
    """Small fallback compatible with the part of gym spaces used here."""

    def __init__(self, n: int):
        self.n = int(n)

    def sample(self) -> int:
        return int(np.random.randint(self.n))


class _Box:
    """Small fallback compatible with the part of gym spaces used here."""

    def __init__(self, low: float, high: float, shape: Tuple[int, ...], dtype: Any):
        self.low = low
        self.high = high
        self.shape = shape
        self.dtype = dtype


## Class Definition
This cell creates the environment shell and static agent metadata. Method groups are attached in later cells so functionality stays separated.


In [ ]:
class MultiAgentHospitalEnv:
    """PettingZoo-style parallel emergency department flow environment.

The environment models queues, capacity, staff availability, and operational
routing decisions. It deliberately avoids grid-world movement, coordinates,
collision, floorplans, and physical navigation.
    """

    metadata = {"name": "MultiAgentHospitalEnv", "is_parallelizable": True}

    possible_agents = [
            "reception_agent",
            "nurse_triage_agent",
            "doctor_agent",
            "physician_agent",
            "radiologist_agent",
            "ambulance_paramedic_agent",
            "administrator_agent",
        ]


## Initialization
Initialisation sets capacities, probabilities, staff schedule, action spaces, observation spaces, and internal patient queues. Key metrics: capacity maxima and arrival probability.


In [ ]:
def __init__(
        self,
        registration_queue_max: int = 20,
        assessment_queue_max: int = 20,
        resus_max: int = 3,
        major_max: int = 8,
        minor_max: int = 10,
        general_max: int = 20,
        icu_max: int = 6,
        max_steps: int = 500,
        arrival_probability: float = 0.65,
        patient_type_probabilities: Tuple[float, float, float] = (0.2, 0.3, 0.5),
        staff_schedule: Optional[Dict[str, Dict[str, int]]] = None,
    ) -> None:
        self.registration_queue_max = int(registration_queue_max)
        self.assessment_queue_max = int(assessment_queue_max)
        self.resus_max = int(resus_max)
        self.major_max = int(major_max)
        self.minor_max = int(minor_max)
        self.general_max = int(general_max)
        self.icu_max = int(icu_max)
        self.max_steps = int(max_steps)
        self.arrival_probability = float(arrival_probability)
        self.patient_type_probabilities = np.asarray(patient_type_probabilities, dtype=float)
        self.patient_type_probabilities /= self.patient_type_probabilities.sum()
        self.staff_schedule = staff_schedule or STAFF_SCHEDULE

        self.resus_release_probability = 0.25
        self.major_release_probability = 0.30
        self.minor_release_probability = 0.50
        self.general_discharge_probability = 0.35
        self.icu_discharge_probability = 0.20

        self.agents: List[str] = list(self.possible_agents)
        self._np_random = np.random.default_rng()
        self._last_actions: Dict[str, int] = {}
        self._admin_mode = 0
        self._step_count = 0

        discrete_cls = spaces.Discrete if spaces is not None else _Discrete
        box_cls = spaces.Box if spaces is not None else _Box
        self.action_spaces = {
            agent: discrete_cls(len(ACTION_MEANINGS[agent])) for agent in self.possible_agents
        }
        self.observation_spaces = {
            agent: box_cls(low=0.0, high=1.0, shape=(18,), dtype=np.float32)
            for agent in self.possible_agents
        }

        self._arrival_queue: Deque[Patient] = deque()
        self._registration_patients: Deque[Patient] = deque()
        self._assessment_patients: Deque[Patient] = deque()
        self._imaging_patients: Deque[Patient] = deque()
        self._treatment_patients: Deque[Patient] = deque()
        self._admission_patients: Deque[Patient] = deque()
        self._resus_patients: Deque[Patient] = deque()
        self._major_patients: Deque[Patient] = deque()
        self._minor_patients: Deque[Patient] = deque()
        self._general_patients: Deque[Patient] = deque()
        self._icu_patients: Deque[Patient] = deque()

        self._set_initial_public_state()

MultiAgentHospitalEnv.__init__ = __init__


## Parallel API
`reset` and `step` follow the PettingZoo-style parallel API. Step metrics include rewards, terminations, truncations, and per-agent infos.


In [ ]:
# Metrics: rewards reflect local role actions plus shared flow pressure; no learning update or epsilon is used.

def reset(
        self, seed: Optional[int] = None, options: Optional[Dict[str, Any]] = None
    ) -> Tuple[Dict[str, np.ndarray], Dict[str, Dict[str, Any]]]:
        """Reset environment and return observations plus infos."""

        if seed is not None:
            self._np_random = np.random.default_rng(seed)

        options = options or {}
        self.agents = list(self.possible_agents)
        self._step_count = 0
        self._admin_mode = 0
        self._last_actions = {agent: 0 for agent in self.possible_agents}

        self._arrival_queue.clear()
        self._registration_patients.clear()
        self._assessment_patients.clear()
        self._imaging_patients.clear()
        self._treatment_patients.clear()
        self._admission_patients.clear()
        self._resus_patients.clear()
        self._major_patients.clear()
        self._minor_patients.clear()
        self._general_patients.clear()
        self._icu_patients.clear()

        self.shift = options.get("shift", "day")
        self._apply_shift_staffing()
        self.resus_free = self.resus_max
        self.major_free = self.major_max
        self.minor_free = self.minor_max
        self.general_free = self.general_max
        self.icu_free = self.icu_max

        initial_patients = int(options.get("initial_patients", 1))
        for _ in range(max(initial_patients, 0)):
            self._arrival_queue.append(self._new_patient())

        self._sync_public_state()
        return self._observations(), self._infos()

def step(
        self, actions: Dict[str, int]
    ) -> Tuple[
        Dict[str, np.ndarray],
        Dict[str, float],
        Dict[str, bool],
        Dict[str, bool],
        Dict[str, Dict[str, Any]],
    ]:
        """Advance one parallel timestep using a dict of agent actions."""

        if not self.agents:
            return {}, {}, {}, {}, {}

        clean_actions = self._clean_actions(actions)
        self._last_actions = clean_actions
        self._step_count += 1

        rewards = {agent: 0.0 for agent in self.agents}
        events: List[str] = []

        self._advance_shift()
        self._apply_shift_staffing()
        self._release_capacity(events, rewards)
        self._maybe_add_patient(events)

        self._apply_ambulance_action(clean_actions["ambulance_paramedic_agent"], rewards, events)
        self._apply_administrator_action(clean_actions["administrator_agent"], rewards, events)
        self._apply_reception_action(clean_actions["reception_agent"], rewards, events)
        self._apply_nurse_action(clean_actions["nurse_triage_agent"], rewards, events)
        self._apply_radiologist_action(clean_actions["radiologist_agent"], rewards, events)
        self._apply_doctor_action(clean_actions["doctor_agent"], rewards, events)
        self._apply_physician_action(clean_actions["physician_agent"], rewards, events)

        congestion_penalty = self._update_waiting_times()
        global_reward = self._global_flow_reward() - congestion_penalty
        for agent in rewards:
            rewards[agent] += global_reward

        self._sync_public_state()
        is_done = False
        is_truncated = self._step_count >= self.max_steps
        terminations = {agent: is_done for agent in self.agents}
        truncations = {agent: is_truncated for agent in self.agents}
        infos = self._infos(events=events)

        if is_done or is_truncated:
            self.agents = []

        return self._observations(), rewards, terminations, truncations, infos

def observation_space(self, agent: str) -> Any:
        return self.observation_spaces[agent]

def action_space(self, agent: str) -> Any:
        return self.action_spaces[agent]

def state(self) -> np.ndarray:
        """Return the shared global state vector used by all agents."""

        return self._observation_vector()



MultiAgentHospitalEnv.reset = reset

MultiAgentHospitalEnv.step = step

MultiAgentHospitalEnv.observation_space = observation_space

MultiAgentHospitalEnv.action_space = action_space

MultiAgentHospitalEnv.state = state


## Public State and Staff Shift
These helpers sync public state variables and update day/evening/night staffing. delta_staffing is determined by shift, not individual movement.


In [ ]:
# Metrics: shift changes alter available staff counts and therefore capacity to process queues.

def _set_initial_public_state(self) -> None:
        self.registration_queue = 0
        self.assessment_queue = 0
        self.resus_free = self.resus_max
        self.major_free = self.major_max
        self.minor_free = self.minor_max
        self.general_free = self.general_max
        self.icu_free = self.icu_max
        self.patient_type = 2
        self.patient_stage = 0
        self.waiting_time = 0
        self.shift = "day"
        self._apply_shift_staffing()

def _apply_shift_staffing(self) -> None:
        staff = self.staff_schedule[self.shift]
        self.nurses_available = staff["nurses"]
        self.doctors_available = staff["doctors"]
        self.physicians_available = staff["physicians"]
        self.radiologists_available = staff["radiologists"]
        self.receptionists_available = staff["receptionists"]
        self.administrators_available = staff["administrators"]
        self.paramedics_available = staff["paramedics"]

def _advance_shift(self) -> None:
        cycle_step = self._step_count % 72
        if cycle_step < 24:
            self.shift = "day"
        elif cycle_step < 48:
            self.shift = "evening"
        else:
            self.shift = "night"



MultiAgentHospitalEnv._set_initial_public_state = _set_initial_public_state

MultiAgentHospitalEnv._apply_shift_staffing = _apply_shift_staffing

MultiAgentHospitalEnv._advance_shift = _advance_shift


## Patient Arrival and Action Cleaning
Patients arrive probabilistically with type probabilities 0.2/0.3/0.5. Invalid actions are clipped into each agent action range.


In [ ]:
# Metrics: arrival probability controls delta_arrival_queue; epsilon is not used for action selection here.

def _new_patient(self, patient_type: Optional[int] = None) -> Patient:
        if patient_type is None:
            patient_type = int(self._np_random.choice([0, 1, 2], p=self.patient_type_probabilities))
        return Patient(patient_type=patient_type, stage=0)

def _clean_actions(self, actions: Dict[str, int]) -> Dict[str, int]:
        clean = {}
        for agent in self.possible_agents:
            action = int(actions.get(agent, 0))
            max_action = len(ACTION_MEANINGS[agent]) - 1
            clean[agent] = min(max(action, 0), max_action)
        return clean

def _maybe_add_patient(self, events: List[str]) -> None:
        if self._np_random.random() < self.arrival_probability:
            self._arrival_queue.append(self._new_patient())
            events.append("probabilistic_patient_arrival")



MultiAgentHospitalEnv._new_patient = _new_patient

MultiAgentHospitalEnv._clean_actions = _clean_actions

MultiAgentHospitalEnv._maybe_add_patient = _maybe_add_patient


## Capacity Release
Beds and treatment areas are released stochastically using configured discharge probabilities. delta_capacity is positive when patients leave occupied areas.


In [ ]:
# Metrics: release probabilities drive delta_resus, delta_major, delta_minor, delta_general, and delta_ICU capacity.

def _release_capacity(self, events: List[str], rewards: Dict[str, float]) -> None:
        released = 0
        released += self._release_from_area(
            self._resus_patients, self.resus_max, "resus", self.resus_release_probability, events
        )
        released += self._release_from_area(
            self._major_patients, self.major_max, "major", self.major_release_probability, events
        )
        released += self._release_from_area(
            self._minor_patients, self.minor_max, "minor", self.minor_release_probability, events
        )
        released += self._release_from_area(
            self._general_patients,
            self.general_max,
            "general",
            self.general_discharge_probability,
            events,
        )
        released += self._release_from_area(
            self._icu_patients, self.icu_max, "icu", self.icu_discharge_probability, events
        )
        if released:
            rewards["administrator_agent"] += 0.2 * released

def _release_from_area(
        self,
        queue: Deque[Patient],
        capacity: int,
        area: str,
        probability: float,
        events: List[str],
    ) -> int:
        remaining: Deque[Patient] = deque()
        released = 0
        while queue:
            patient = queue.popleft()
            if self._np_random.random() < probability:
                patient.stage = 11
                patient.area = None
                self._remove_patient_from_operational_queues(patient)
                released += 1
            else:
                remaining.append(patient)
        queue.extend(remaining)
        setattr(self, f"{area}_free", capacity - len(queue))
        if released:
            events.append(f"{area}_release_{released}")
        return released



MultiAgentHospitalEnv._release_capacity = _release_capacity

MultiAgentHospitalEnv._release_from_area = _release_from_area


## Ambulance, Administrator, and Reception Actions
These handlers manage incoming emergencies, global flow pressure, and registration movement. Metrics include delta_arrival_queue and delta_registration_queue.


In [ ]:
# Metrics: these actions change queues, emergency priority, discharge pressure, and local rewards.

def _apply_ambulance_action(
        self, action: int, rewards: Dict[str, float], events: List[str]
    ) -> None:
        if action == 1 and self.paramedics_available > 0:
            self._arrival_queue.appendleft(self._new_patient(patient_type=0))
            rewards["ambulance_paramedic_agent"] += 1.0
            events.append("ambulance_brought_emergency_patient")
        elif action == 2:
            patient = self._find_patient(self._arrival_queue, allowed_types={2})
            if patient is not None:
                rewards["ambulance_paramedic_agent"] += 0.5
                events.append("noncritical_patient_redirected")
            else:
                rewards["ambulance_paramedic_agent"] -= 0.1

def _apply_administrator_action(
        self, action: int, rewards: Dict[str, float], events: List[str]
    ) -> None:
        self._admin_mode = action
        if action == 1:
            self._prioritise_emergency_queues()
            rewards["administrator_agent"] += 0.4
            events.append("administrator_prioritised_emergency_flow")
        elif action == 2:
            self.assessment_queue_max += 1
            rewards["administrator_agent"] += 0.2
            events.append("administrator_prioritised_general_throughput")
        elif action == 3:
            rewards["administrator_agent"] += 0.3 if self.icu_free > 0 else -0.3
            events.append("administrator_preserved_icu_capacity")
        elif action == 4:
            released = self._force_one_low_acuity_discharge()
            rewards["administrator_agent"] += 0.8 if released else -0.2
            events.append("administrator_increased_discharge_pressure")

def _apply_reception_action(
        self, action: int, rewards: Dict[str, float], events: List[str]
    ) -> None:
        if action == 1:
            if (
                self._arrival_queue
                and len(self._registration_patients) < self.registration_queue_max
                and self.receptionists_available > 0
            ):
                patient = self._pop_priority(self._arrival_queue)
                patient.stage = 1
                self._registration_patients.append(patient)
                rewards["reception_agent"] += 0.6
                events.append("patient_registered")
            else:
                rewards["reception_agent"] -= 0.2
        elif action == 2:
            if self._registration_patients and len(self._assessment_patients) < self.assessment_queue_max:
                patient = self._pop_priority(self._registration_patients)
                patient.stage = 2
                self._assessment_patients.append(patient)
                rewards["reception_agent"] += 0.8
                events.append("patient_moved_to_assessment")
            else:
                rewards["reception_agent"] -= 0.2



MultiAgentHospitalEnv._apply_ambulance_action = _apply_ambulance_action

MultiAgentHospitalEnv._apply_administrator_action = _apply_administrator_action

MultiAgentHospitalEnv._apply_reception_action = _apply_reception_action


## Nurse, Radiologist, Doctor, and Physician Actions
These handlers route, process, treat, escalate, admit, discharge, or delay patients. Metrics include waiting time, delta_stage, and bed usage.


In [ ]:
# Metrics: clinical actions affect delta_stage, delta_capacity, waiting time, and role-specific rewards.

def _apply_nurse_action(
        self, action: int, rewards: Dict[str, float], events: List[str]
    ) -> None:
        if action == 0:
            return
        if action == 4:
            self._delay_queue(self._assessment_patients)
            rewards["nurse_triage_agent"] -= 0.5
            events.append("assessment_delayed")
            return
        if not self._assessment_patients or self.nurses_available <= 0:
            rewards["nurse_triage_agent"] -= 0.2
            return

        patient = self._pop_priority(self._assessment_patients)
        target = {1: "minor", 2: "major", 3: "resus"}[action]
        if self._allocate_area(patient, target):
            patient.stage = {"minor": 8, "major": 7, "resus": 6}[target]
            self._treatment_patients.append(patient)
            rewards["nurse_triage_agent"] += self._triage_reward(patient, target)
            events.append(f"patient_sent_to_{target}")
        else:
            patient.stage = 12
            patient.waiting_time += 1
            self._assessment_patients.appendleft(patient)
            rewards["nurse_triage_agent"] -= 1.0
            events.append(f"{target}_capacity_unavailable")

def _apply_radiologist_action(
        self, action: int, rewards: Dict[str, float], events: List[str]
    ) -> None:
        if action == 1:
            if self._imaging_patients and self.radiologists_available > 0:
                patient = self._pop_priority(self._imaging_patients)
                patient.needs_imaging = False
                patient.stage = 5
                self._treatment_patients.append(patient)
                rewards["radiologist_agent"] += 0.7
                events.append("imaging_processed")
            else:
                rewards["radiologist_agent"] -= 0.2
        elif action == 2:
            self._delay_queue(self._imaging_patients)
            rewards["radiologist_agent"] -= 0.4
            events.append("imaging_delayed")

def _apply_doctor_action(
        self, action: int, rewards: Dict[str, float], events: List[str]
    ) -> None:
        if action == 0:
            return
        if action == 5:
            self._delay_queue(self._treatment_patients)
            rewards["doctor_agent"] -= 0.5
            events.append("treatment_delayed")
            return
        if not self._treatment_patients or self.doctors_available <= 0:
            rewards["doctor_agent"] -= 0.2
            return

        patient = self._pop_priority(self._treatment_patients)
        if action == 1:
            patient.stage = 5
            if patient.patient_type == 2:
                patient.stage = 11
                self._release_patient_area(patient)
                rewards["doctor_agent"] += 1.0
                events.append("minor_patient_treated_and_discharged")
            else:
                self._admission_patients.append(patient)
                rewards["doctor_agent"] += 0.8
                events.append("patient_treated_for_admission_decision")
        elif action == 2:
            patient.stage = 4
            patient.needs_imaging = True
            self._imaging_patients.append(patient)
            rewards["doctor_agent"] += 0.4
            events.append("imaging_requested")
        elif action == 3:
            escalated = self._escalate_patient(patient)
            rewards["doctor_agent"] += 0.6 if escalated else -0.5
            events.append("patient_escalated" if escalated else "escalation_blocked")
        elif action == 4:
            deescalated = self._deescalate_patient(patient)
            rewards["doctor_agent"] += 0.5 if deescalated else -0.3
            events.append("patient_deescalated" if deescalated else "deescalation_blocked")

def _apply_physician_action(
        self, action: int, rewards: Dict[str, float], events: List[str]
    ) -> None:
        if action == 0:
            return
        if action == 4:
            self._delay_queue(self._admission_patients)
            rewards["physician_agent"] -= 0.5
            events.append("admission_delayed")
            return
        if not self._admission_patients or self.physicians_available <= 0:
            rewards["physician_agent"] -= 0.2
            return

        patient = self._pop_priority(self._admission_patients)
        if action == 1:
            if self.general_free > 0:
                patient.stage = 9
                self._release_patient_area(patient)
                patient.area = "general"
                self._general_patients.append(patient)
                self.general_free -= 1
                rewards["physician_agent"] += 0.8
                events.append("patient_admitted_general")
            else:
                self._block_admission(patient, rewards, "physician_agent", events, "general")
        elif action == 2:
            if self.icu_free > 0 and not (self._admin_mode == 3 and patient.patient_type == 2):
                patient.stage = 10
                self._release_patient_area(patient)
                patient.area = "icu"
                self._icu_patients.append(patient)
                self.icu_free -= 1
                rewards["physician_agent"] += 1.2 if patient.patient_type == 0 else 0.4
                events.append("patient_admitted_icu")
            else:
                self._block_admission(patient, rewards, "physician_agent", events, "icu")
        elif action == 3:
            patient.stage = 11
            self._release_patient_area(patient)
            rewards["physician_agent"] += 0.8 if patient.patient_type == 2 else -0.3
            events.append("patient_discharged_by_physician")



MultiAgentHospitalEnv._apply_nurse_action = _apply_nurse_action

MultiAgentHospitalEnv._apply_radiologist_action = _apply_radiologist_action

MultiAgentHospitalEnv._apply_doctor_action = _apply_doctor_action

MultiAgentHospitalEnv._apply_physician_action = _apply_physician_action


## Allocation and Escalation Helpers
These helpers move patients between minor, major, resus, general, ICU, and discharge states. delta_capacity is updated whenever area occupancy changes.


In [ ]:
# Metrics: allocation changes free-bed counts and rewards triage correctness by patient acuity.

def _allocate_area(self, patient: Patient, target: str) -> bool:
        free_attr = f"{target}_free"
        if getattr(self, free_attr) <= 0:
            return False
        setattr(self, free_attr, getattr(self, free_attr) - 1)
        patient.area = target
        if target == "resus":
            self._resus_patients.append(patient)
        elif target == "major":
            self._major_patients.append(patient)
        elif target == "minor":
            self._minor_patients.append(patient)
        return True

def _triage_reward(self, patient: Patient, target: str) -> float:
        if patient.patient_type == 0 and target == "resus":
            return 1.2
        if patient.patient_type == 1 and target == "major":
            return 1.0
        if patient.patient_type == 2 and target == "minor":
            return 1.0
        if patient.patient_type == 0 and target == "minor":
            return -1.0
        return -0.2

def _escalate_patient(self, patient: Patient) -> bool:
        if patient.area == "minor" and self.major_free > 0:
            self._move_area(patient, "minor", "major", 7)
            self._treatment_patients.append(patient)
            return True
        if patient.area == "major" and self.resus_free > 0:
            self._move_area(patient, "major", "resus", 6)
            self._treatment_patients.append(patient)
            return True
        self._treatment_patients.append(patient)
        return False

def _deescalate_patient(self, patient: Patient) -> bool:
        if patient.area == "resus" and self.major_free > 0:
            self._move_area(patient, "resus", "major", 7)
            self._treatment_patients.append(patient)
            return True
        if patient.area == "major" and self.minor_free > 0:
            self._move_area(patient, "major", "minor", 8)
            self._treatment_patients.append(patient)
            return True
        self._treatment_patients.append(patient)
        return False

def _move_area(self, patient: Patient, old_area: str, new_area: str, new_stage: int) -> None:
        self._remove_from_area_queue(patient, old_area)
        setattr(self, f"{old_area}_free", getattr(self, f"{old_area}_free") + 1)
        setattr(self, f"{new_area}_free", getattr(self, f"{new_area}_free") - 1)
        getattr(self, f"_{new_area}_patients").append(patient)
        patient.area = new_area
        patient.stage = new_stage

def _release_patient_area(self, patient: Patient) -> None:
        if patient.area in {"resus", "major", "minor"}:
            self._remove_from_area_queue(patient, patient.area)
            free_attr = f"{patient.area}_free"
            max_attr = f"{patient.area}_max"
            setattr(self, free_attr, min(getattr(self, free_attr) + 1, getattr(self, max_attr)))
            patient.area = None

def _remove_from_area_queue(self, patient: Patient, area: str) -> None:
        queue = getattr(self, f"_{area}_patients")
        try:
            queue.remove(patient)
        except ValueError:
            pass

def _remove_patient_from_operational_queues(self, patient: Patient) -> None:
        for queue in [
            self._imaging_patients,
            self._treatment_patients,
            self._admission_patients,
        ]:
            try:
                queue.remove(patient)
            except ValueError:
                pass

def _block_admission(
        self,
        patient: Patient,
        rewards: Dict[str, float],
        agent: str,
        events: List[str],
        bed_type: str,
    ) -> None:
        patient.stage = 12
        patient.waiting_time += 1
        self._admission_patients.appendleft(patient)
        rewards[agent] -= 0.8
        events.append(f"{bed_type}_admission_blocked")



MultiAgentHospitalEnv._allocate_area = _allocate_area

MultiAgentHospitalEnv._triage_reward = _triage_reward

MultiAgentHospitalEnv._escalate_patient = _escalate_patient

MultiAgentHospitalEnv._deescalate_patient = _deescalate_patient

MultiAgentHospitalEnv._move_area = _move_area

MultiAgentHospitalEnv._release_patient_area = _release_patient_area

MultiAgentHospitalEnv._remove_from_area_queue = _remove_from_area_queue

MultiAgentHospitalEnv._remove_patient_from_operational_queues = _remove_patient_from_operational_queues

MultiAgentHospitalEnv._block_admission = _block_admission


## Queue Policy and Shared Reward
These helpers prioritise emergencies, delay queues, and compute shared flow pressure. Queue pressure penalises congestion and long waits.


In [ ]:
# Metrics: waiting_time and queue_pressure shape the shared reward; no PPO delta is computed here.

def _force_one_low_acuity_discharge(self) -> bool:
        for queue, area in [
            (self._minor_patients, "minor"),
            (self._general_patients, "general"),
            (self._major_patients, "major"),
        ]:
            patient = self._find_patient(queue, allowed_types={2})
            if patient is not None:
                patient.stage = 11
                free_attr = f"{area}_free"
                max_attr = f"{area}_max"
                setattr(self, free_attr, min(getattr(self, free_attr) + 1, getattr(self, max_attr)))
                return True
        return False

def _find_patient(self, queue: Deque[Patient], allowed_types: Iterable[int]) -> Optional[Patient]:
        allowed = set(allowed_types)
        for patient in list(queue):
            if patient.patient_type in allowed:
                queue.remove(patient)
                return patient
        return None

def _pop_priority(self, queue: Deque[Patient]) -> Patient:
        if self._admin_mode == 1:
            for patient in list(queue):
                if patient.patient_type == 0:
                    queue.remove(patient)
                    return patient
        return queue.popleft()

def _prioritise_emergency_queues(self) -> None:
        for queue in [
            self._arrival_queue,
            self._registration_patients,
            self._assessment_patients,
            self._imaging_patients,
            self._treatment_patients,
            self._admission_patients,
        ]:
            ordered = sorted(queue, key=lambda patient: patient.patient_type)
            queue.clear()
            queue.extend(ordered)

def _delay_queue(self, queue: Deque[Patient]) -> None:
        if queue:
            patient = queue[0]
            patient.stage = 12
            patient.waiting_time += 1

def _update_waiting_times(self) -> float:
        penalty = 0.0
        for queue in self._active_waiting_queues():
            for patient in queue:
                patient.waiting_time += 1
                if patient.patient_type == 0 and patient.waiting_time > 3:
                    penalty += 0.4
                elif patient.waiting_time > 6:
                    penalty += 0.2
        congestion = (
            len(self._registration_patients) / max(self.registration_queue_max, 1)
            + len(self._assessment_patients) / max(self.assessment_queue_max, 1)
        )
        return penalty + 0.2 * congestion

def _active_waiting_queues(self) -> List[Deque[Patient]]:
        return [
            self._arrival_queue,
            self._registration_patients,
            self._assessment_patients,
            self._imaging_patients,
            self._treatment_patients,
            self._admission_patients,
        ]

def _global_flow_reward(self) -> float:
        free_critical = (self.resus_free / max(self.resus_max, 1)) + (
            self.icu_free / max(self.icu_max, 1)
        )
        queue_pressure = len(self._arrival_queue) + len(self._registration_patients)
        return 0.05 * free_critical - 0.03 * queue_pressure



MultiAgentHospitalEnv._force_one_low_acuity_discharge = _force_one_low_acuity_discharge

MultiAgentHospitalEnv._find_patient = _find_patient

MultiAgentHospitalEnv._pop_priority = _pop_priority

MultiAgentHospitalEnv._prioritise_emergency_queues = _prioritise_emergency_queues

MultiAgentHospitalEnv._delay_queue = _delay_queue

MultiAgentHospitalEnv._update_waiting_times = _update_waiting_times

MultiAgentHospitalEnv._active_waiting_queues = _active_waiting_queues

MultiAgentHospitalEnv._global_flow_reward = _global_flow_reward


## Observations and Infos
The observation vector normalises the requested state variables for all agents. Infos expose raw metrics and internal queue lengths for debugging.


In [ ]:
# Metrics: observations include normalised capacity, staffing, patient type/stage, waiting time, and shift.

def _sync_public_state(self) -> None:
        self.registration_queue = len(self._registration_patients)
        self.assessment_queue = len(self._assessment_patients)
        focus = self._focus_patient()
        self.patient_type = focus.patient_type if focus is not None else 2
        self.patient_stage = focus.stage if focus is not None else 11
        self.waiting_time = focus.waiting_time if focus is not None else 0

def _focus_patient(self) -> Optional[Patient]:
        for queue in self._active_waiting_queues():
            if queue:
                return min(queue, key=lambda patient: (patient.patient_type, -patient.waiting_time))
        for queue in [
            self._resus_patients,
            self._major_patients,
            self._minor_patients,
            self._general_patients,
            self._icu_patients,
        ]:
            if queue:
                return queue[0]
        return None

def _observations(self) -> Dict[str, np.ndarray]:
        obs = self._observation_vector()
        return {agent: obs.copy() for agent in self.agents}

def _observation_vector(self) -> np.ndarray:
        shift_index = {"day": 0, "evening": 1, "night": 2}[self.shift]
        max_staff = self.staff_schedule["day"]
        return np.asarray(
            [
                self.registration_queue / max(self.registration_queue_max, 1),
                self.assessment_queue / max(self.assessment_queue_max, 1),
                self.resus_free / max(self.resus_max, 1),
                self.major_free / max(self.major_max, 1),
                self.minor_free / max(self.minor_max, 1),
                self.general_free / max(self.general_max, 1),
                self.icu_free / max(self.icu_max, 1),
                self.patient_type / 2.0,
                self.patient_stage / 12.0,
                min(self.waiting_time, 50) / 50.0,
                self.nurses_available / max(max_staff["nurses"], 1),
                self.doctors_available / max(max_staff["doctors"], 1),
                self.physicians_available / max(max_staff["physicians"], 1),
                self.radiologists_available / max(max_staff["radiologists"], 1),
                self.receptionists_available / max(max_staff["receptionists"], 1),
                self.administrators_available / max(max_staff["administrators"], 1),
                self.paramedics_available / max(max_staff["paramedics"], 1),
                shift_index / 2.0,
            ],
            dtype=np.float32,
        )

def _infos(self, events: Optional[List[str]] = None) -> Dict[str, Dict[str, Any]]:
        shared_info = {
            "step": self._step_count,
            "shift": self.shift,
            "events": events or [],
            "action_meanings": ACTION_MEANINGS,
            "patient_type_meanings": PATIENT_TYPES,
            "patient_stage_meanings": PATIENT_STAGES,
            "raw_state": {
                "registration_queue": self.registration_queue,
                "assessment_queue": self.assessment_queue,
                "resus_free": self.resus_free,
                "major_free": self.major_free,
                "minor_free": self.minor_free,
                "general_free": self.general_free,
                "icu_free": self.icu_free,
                "patient_type": self.patient_type,
                "patient_stage": self.patient_stage,
                "waiting_time": self.waiting_time,
                "nurses_available": self.nurses_available,
                "doctors_available": self.doctors_available,
                "physicians_available": self.physicians_available,
                "radiologists_available": self.radiologists_available,
                "receptionists_available": self.receptionists_available,
                "administrators_available": self.administrators_available,
                "paramedics_available": self.paramedics_available,
                "shift": self.shift,
            },
            "internal_queue_lengths": {
                "arrival": len(self._arrival_queue),
                "registration": len(self._registration_patients),
                "assessment": len(self._assessment_patients),
                "imaging": len(self._imaging_patients),
                "treatment": len(self._treatment_patients),
                "admission": len(self._admission_patients),
                "resus": len(self._resus_patients),
                "major": len(self._major_patients),
                "minor": len(self._minor_patients),
                "general": len(self._general_patients),
                "icu": len(self._icu_patients),
            },
        }
        return {agent: dict(shared_info) for agent in self.agents}



MultiAgentHospitalEnv._sync_public_state = _sync_public_state

MultiAgentHospitalEnv._focus_patient = _focus_patient

MultiAgentHospitalEnv._observations = _observations

MultiAgentHospitalEnv._observation_vector = _observation_vector

MultiAgentHospitalEnv._infos = _infos


## Optional API Smoke Check
This runs one reset and one random-action step only. It verifies API shape without MAPPO, training, neural networks, delta_ updates, or delta_-greedy learning.


In [ ]:
# No training occurs here: this is one reset and one environment step only.
# epsilon is not used in the environment; exploration belongs to a later MAPPO policy.
env = MultiAgentHospitalEnv()
observations, infos = env.reset(seed=42)
actions = {agent: env.action_space(agent).sample() for agent in env.agents}
observations, rewards, terminations, truncations, infos = env.step(actions)

print("agents:", env.possible_agents)
print("observation_shape:", next(iter(observations.values())).shape)
print("rewards:", rewards)


## MAPPO Template Imports
This follows the template import style but guards PyTorch. Graphing works without training; MAPPO training needs PyTorch to import correctly.


In [ ]:
# Template-style MAPPO imports. PyTorch is optional until training is actually enabled.
import os
import json
import math
import matplotlib.pyplot as plt

try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.distributions import Categorical
    TORCH_AVAILABLE = True
    TORCH_IMPORT_ERROR = None
except Exception as exc:
    torch = None
    nn = None
    optim = None
    Categorical = None
    TORCH_AVAILABLE = False
    TORCH_IMPORT_ERROR = exc
    print(f"PyTorch unavailable in this runtime: {exc}")


## Device Detection
The device cell mirrors the template and selects CUDA, MPS, or CPU. If PyTorch fails locally, training cells stay disabled.


In [ ]:
# Device Detection, matching the template format.
if TORCH_AVAILABLE:
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using CUDA")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using MPS")
    else:
        device = torch.device("cpu")
        print("Using CPU")
else:
    device = None
    print("Training cells require a working PyTorch install before MAPPO can run.")


## Actor and Centralized Critic
Each agent gets an Actor policy, while the critic reads the concatenated hospital state. No network is trained unless `RUN_MAPPO_TRAINING=True`.


In [ ]:
def _require_torch():
    """Raise a clear error only if a training cell is used without PyTorch."""
    if not TORCH_AVAILABLE:
        raise RuntimeError(f"PyTorch is required for MAPPO training: {TORCH_IMPORT_ERROR}")


if TORCH_AVAILABLE:
    class Actor(nn.Module):
        """Per-agent policy network: observation -> action probabilities."""

        def __init__(self, obs_dim, action_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(obs_dim, 128), nn.Tanh(),
                nn.Linear(128, 128), nn.Tanh(),
                nn.Linear(128, action_dim),
                nn.Softmax(dim=-1),
            )

        def forward(self, x):
            return self.net(x)


    class CentralizedCritic(nn.Module):
        """Central critic: concatenated hospital observations -> shared value."""

        def __init__(self, total_obs_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(total_obs_dim, 256), nn.Tanh(),
                nn.Linear(256, 256), nn.Tanh(),
                nn.Linear(256, 1),
            )

        def forward(self, x):
            return self.net(x)
else:
    class Actor:
        def __init__(self, *args, **kwargs):
            _require_torch()


    class CentralizedCritic:
        def __init__(self, *args, **kwargs):
            _require_torch()


## GAE and Metric Helpers
`compute_gae` calculates PPO delta, advantages, and returns. Metric helpers store reward, waiting, queue, loss, entropy, and free-bed traces.


In [ ]:
def compute_gae(rewards, dones, values, gamma=0.99, gae_lambda=0.95):
    """Compute PPO returns and advantages using GAE delta."""
    _require_torch()
    rewards = list(rewards)
    dones = list(dones)
    values = list(values) + [0.0]
    advantages = []
    gae = 0.0

    for t in reversed(range(len(rewards))):
        mask = 1.0 - float(dones[t])
        delta = rewards[t] + gamma * values[t + 1] * mask - values[t]
        gae = delta + gamma * gae_lambda * mask * gae
        advantages.insert(0, gae)

    returns = [adv + val for adv, val in zip(advantages, values[:-1])]
    return (
        torch.tensor(returns, dtype=torch.float32, device=device),
        torch.tensor(advantages, dtype=torch.float32, device=device),
    )


def global_observation(obs_dict, agents):
    """Concatenate all agent observations for the central critic."""
    return np.hstack([obs_dict[agent] for agent in agents]).astype(np.float32)


def empty_metric_history():
    """Create metric storage for rewards, losses, queues, waiting time, and beds."""
    return {
        "episode": [],
        "team_reward": [],
        "actor_loss": [],
        "critic_loss": [],
        "entropy": [],
        "mean_waiting_time": [],
        "mean_registration_queue": [],
        "mean_assessment_queue": [],
        "mean_icu_free": [],
        "mean_general_free": [],
    }


## Training Function
This is the template MAPPO training function adapted to `MultiAgentHospitalEnv`. It uses eps_clip, GAE lambda, entropy, and centralised critic loss.


In [ ]:
def train_mappo(
    episodes=250,
    ppo_epochs=5,
    eps_clip=0.2,
    gamma=0.99,
    gae_lambda=0.95,
    actor_lr=3e-4,
    critic_lr=1e-3,
    entropy_coef=0.01,
    value_coef=0.5,
    max_steps=100,
    seed=42,
    print_every=25,
):
    """Train MAPPO on MultiAgentHospitalEnv using the template structure."""
    _require_torch()
    env = MultiAgentHospitalEnv(max_steps=max_steps)
    obs_dict, _ = env.reset(seed=seed)
    agents = list(env.possible_agents)

    actors = {
        agent: Actor(env.observation_space(agent).shape[0], env.action_space(agent).n).to(device)
        for agent in agents
    }
    total_obs_dim = sum(env.observation_space(agent).shape[0] for agent in agents)
    critic = CentralizedCritic(total_obs_dim).to(device)

    actor_params = []
    for actor in actors.values():
        actor_params.extend(list(actor.parameters()))
    optimizer = optim.Adam(
        [
            {"params": actor_params, "lr": actor_lr},
            {"params": critic.parameters(), "lr": critic_lr},
        ]
    )

    metrics = empty_metric_history()

    for ep in range(episodes):
        obs_dict, _ = env.reset(seed=None if seed is None else seed + ep)
        data = {
            agent: {"obs": [], "global_obs": [], "actions": [], "log_probs": [], "rewards": [], "values": [], "dones": []}
            for agent in agents
        }
        episode_team_reward = 0.0
        waiting_trace, reg_trace, assess_trace, icu_trace, general_trace = [], [], [], [], []

        while env.agents:
            g_obs = global_observation(obs_dict, agents)
            g_obs_t = torch.tensor(g_obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                value = critic(g_obs_t).squeeze().item()

            actions = {}
            for agent in agents:
                obs = np.asarray(obs_dict[agent], dtype=np.float32)
                obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    probs = actors[agent](obs_t)
                    dist = Categorical(probs)
                    action = dist.sample()
                    log_prob = dist.log_prob(action)

                actions[agent] = int(action.item())
                data[agent]["obs"].append(obs)
                data[agent]["global_obs"].append(g_obs)
                data[agent]["actions"].append(actions[agent])
                data[agent]["log_probs"].append(float(log_prob.item()))
                data[agent]["values"].append(float(value))

            next_obs_dict, reward_dict, term_dict, trunc_dict, info_dict = env.step(actions)
            team_reward = float(np.mean([reward_dict[agent] for agent in agents]))
            episode_team_reward += team_reward

            for agent in agents:
                done = bool(term_dict.get(agent, False) or trunc_dict.get(agent, False))
                data[agent]["rewards"].append(team_reward)
                data[agent]["dones"].append(done)

            if info_dict:
                raw_state = next(iter(info_dict.values()))["raw_state"]
                waiting_trace.append(raw_state["waiting_time"])
                reg_trace.append(raw_state["registration_queue"])
                assess_trace.append(raw_state["assessment_queue"])
                icu_trace.append(raw_state["icu_free"])
                general_trace.append(raw_state["general_free"])

            obs_dict = next_obs_dict

        actor_loss_value = 0.0
        critic_loss_value = 0.0
        entropy_value = 0.0

        for _ in range(ppo_epochs):
            optimizer.zero_grad()
            actor_losses, critic_losses, entropies = [], [], []

            for agent in agents:
                if not data[agent]["obs"]:
                    continue

                returns, advantages = compute_gae(
                    data[agent]["rewards"],
                    data[agent]["dones"],
                    data[agent]["values"],
                    gamma=gamma,
                    gae_lambda=gae_lambda,
                )
                advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)

                states = torch.tensor(np.asarray(data[agent]["obs"]), dtype=torch.float32, device=device)
                global_states = torch.tensor(np.asarray(data[agent]["global_obs"]), dtype=torch.float32, device=device)
                actions_t = torch.tensor(data[agent]["actions"], dtype=torch.long, device=device)
                old_log_probs = torch.tensor(data[agent]["log_probs"], dtype=torch.float32, device=device)

                probs = actors[agent](states)
                dist = Categorical(probs)
                new_log_probs = dist.log_prob(actions_t)
                entropy = dist.entropy().mean()

                ratios = torch.exp(new_log_probs - old_log_probs)
                unclipped = ratios * advantages
                clipped = torch.clamp(ratios, 1.0 - eps_clip, 1.0 + eps_clip) * advantages
                actor_loss = -torch.min(unclipped, clipped).mean()

                values = critic(global_states).squeeze(-1)
                critic_loss = nn.functional.mse_loss(values, returns)

                actor_losses.append(actor_loss)
                critic_losses.append(critic_loss)
                entropies.append(entropy)

            if actor_losses:
                loss = (
                    torch.stack(actor_losses).mean()
                    + value_coef * torch.stack(critic_losses).mean()
                    - entropy_coef * torch.stack(entropies).mean()
                )
                loss.backward()
                torch.nn.utils.clip_grad_norm_(actor_params + list(critic.parameters()), max_norm=0.5)
                optimizer.step()

                actor_loss_value = float(torch.stack(actor_losses).mean().detach().cpu())
                critic_loss_value = float(torch.stack(critic_losses).mean().detach().cpu())
                entropy_value = float(torch.stack(entropies).mean().detach().cpu())

        metrics["episode"].append(ep)
        metrics["team_reward"].append(episode_team_reward)
        metrics["actor_loss"].append(actor_loss_value)
        metrics["critic_loss"].append(critic_loss_value)
        metrics["entropy"].append(entropy_value)
        metrics["mean_waiting_time"].append(float(np.mean(waiting_trace)) if waiting_trace else 0.0)
        metrics["mean_registration_queue"].append(float(np.mean(reg_trace)) if reg_trace else 0.0)
        metrics["mean_assessment_queue"].append(float(np.mean(assess_trace)) if assess_trace else 0.0)
        metrics["mean_icu_free"].append(float(np.mean(icu_trace)) if icu_trace else 0.0)
        metrics["mean_general_free"].append(float(np.mean(general_trace)) if general_trace else 0.0)

        if print_every and ep % print_every == 0:
            print(
                f"Episode {ep}: reward={episode_team_reward:.2f}, "
                f"waiting={metrics['mean_waiting_time'][-1]:.2f}, "
                f"actor_loss={actor_loss_value:.4f}, critic_loss={critic_loss_value:.4f}"
            )

    return actors, critic, metrics


## Evaluation and Random Baseline
Random baseline gives immediate graphs without training. Greedy evaluation can be used after training actors are available.


In [ ]:
def run_random_policy_baseline(episodes=10, max_steps=100, seed=42):
    """Run random actions to produce environment graphs without training."""
    env = MultiAgentHospitalEnv(max_steps=max_steps)
    agents = list(env.possible_agents)
    metrics = empty_metric_history()

    for ep in range(episodes):
        obs_dict, _ = env.reset(seed=seed + ep)
        episode_team_reward = 0.0
        waiting_trace, reg_trace, assess_trace, icu_trace, general_trace = [], [], [], [], []

        while env.agents:
            actions = {agent: env.action_space(agent).sample() for agent in env.agents}
            obs_dict, reward_dict, term_dict, trunc_dict, info_dict = env.step(actions)
            episode_team_reward += float(np.mean([reward_dict[agent] for agent in agents]))

            if info_dict:
                raw_state = next(iter(info_dict.values()))["raw_state"]
                waiting_trace.append(raw_state["waiting_time"])
                reg_trace.append(raw_state["registration_queue"])
                assess_trace.append(raw_state["assessment_queue"])
                icu_trace.append(raw_state["icu_free"])
                general_trace.append(raw_state["general_free"])

        metrics["episode"].append(ep)
        metrics["team_reward"].append(episode_team_reward)
        metrics["actor_loss"].append(0.0)
        metrics["critic_loss"].append(0.0)
        metrics["entropy"].append(0.0)
        metrics["mean_waiting_time"].append(float(np.mean(waiting_trace)) if waiting_trace else 0.0)
        metrics["mean_registration_queue"].append(float(np.mean(reg_trace)) if reg_trace else 0.0)
        metrics["mean_assessment_queue"].append(float(np.mean(assess_trace)) if assess_trace else 0.0)
        metrics["mean_icu_free"].append(float(np.mean(icu_trace)) if icu_trace else 0.0)
        metrics["mean_general_free"].append(float(np.mean(general_trace)) if general_trace else 0.0)

    print(f"Random baseline complete for {episodes} episodes.")
    return metrics


def evaluate_mappo_policy(actors, episodes=5, max_steps=100, seed=100):
    """Evaluate trained actors greedily and collect the same metrics."""
    _require_torch()
    env = MultiAgentHospitalEnv(max_steps=max_steps)
    agents = list(env.possible_agents)
    metrics = empty_metric_history()

    for ep in range(episodes):
        obs_dict, _ = env.reset(seed=seed + ep)
        episode_team_reward = 0.0
        waiting_trace, reg_trace, assess_trace, icu_trace, general_trace = [], [], [], [], []

        while env.agents:
            actions = {}
            for agent in env.agents:
                obs_t = torch.tensor(obs_dict[agent], dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    probs = actors[agent](obs_t)
                actions[agent] = int(torch.argmax(probs, dim=-1).item())

            obs_dict, reward_dict, term_dict, trunc_dict, info_dict = env.step(actions)
            episode_team_reward += float(np.mean([reward_dict[agent] for agent in agents]))

            if info_dict:
                raw_state = next(iter(info_dict.values()))["raw_state"]
                waiting_trace.append(raw_state["waiting_time"])
                reg_trace.append(raw_state["registration_queue"])
                assess_trace.append(raw_state["assessment_queue"])
                icu_trace.append(raw_state["icu_free"])
                general_trace.append(raw_state["general_free"])

        metrics["episode"].append(ep)
        metrics["team_reward"].append(episode_team_reward)
        metrics["actor_loss"].append(0.0)
        metrics["critic_loss"].append(0.0)
        metrics["entropy"].append(0.0)
        metrics["mean_waiting_time"].append(float(np.mean(waiting_trace)) if waiting_trace else 0.0)
        metrics["mean_registration_queue"].append(float(np.mean(reg_trace)) if reg_trace else 0.0)
        metrics["mean_assessment_queue"].append(float(np.mean(assess_trace)) if assess_trace else 0.0)
        metrics["mean_icu_free"].append(float(np.mean(icu_trace)) if icu_trace else 0.0)
        metrics["mean_general_free"].append(float(np.mean(general_trace)) if general_trace else 0.0)

    print(f"Greedy MAPPO evaluation complete for {episodes} episodes.")
    return metrics


## Plotting and Saving
Plots replace video rendering because this is an operational flow model, not a spatial environment. Metrics can be saved as JSON.


In [ ]:
def plot_training_metrics(metrics, title="Hospital MAPPO Metrics"):
    """Plot rewards, queue/waiting metrics, losses, and free-bed trends."""
    if not metrics or not metrics.get("episode"):
        raise ValueError("No metrics available to plot.")

    episodes = metrics["episode"]
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle(title)

    axes[0, 0].plot(episodes, metrics["team_reward"], label="team reward")
    axes[0, 0].set_title("Episode Reward")
    axes[0, 0].set_xlabel("Episode")
    axes[0, 0].set_ylabel("Reward")
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].plot(episodes, metrics["mean_waiting_time"], label="waiting time")
    axes[0, 1].plot(episodes, metrics["mean_registration_queue"], label="registration queue")
    axes[0, 1].plot(episodes, metrics["mean_assessment_queue"], label="assessment queue")
    axes[0, 1].set_title("Flow Pressure")
    axes[0, 1].set_xlabel("Episode")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].plot(episodes, metrics["actor_loss"], label="actor loss")
    axes[1, 0].plot(episodes, metrics["critic_loss"], label="critic loss")
    axes[1, 0].plot(episodes, metrics["entropy"], label="entropy")
    axes[1, 0].set_title("MAPPO Training Metrics")
    axes[1, 0].set_xlabel("Episode")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].plot(episodes, metrics["mean_icu_free"], label="ICU free")
    axes[1, 1].plot(episodes, metrics["mean_general_free"], label="general free")
    axes[1, 1].set_title("Free Bed Capacity")
    axes[1, 1].set_xlabel("Episode")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


def save_metrics(metrics, filename="hospital_mappo_metrics.json"):
    """Save metric history to JSON instead of a video, because this is not a spatial renderer."""
    os.makedirs("outputs", exist_ok=True)
    path = os.path.join("outputs", filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved to {path}")
    return path


## Lets Run: Random Baseline Graph
This cell prints a graph like the earlier coursework notebooks. It does not train MAPPO or update neural-network weights.


In [ ]:
# Random baseline graph: safe to run because it does not train a model.
RUN_RANDOM_BASELINE = True

if RUN_RANDOM_BASELINE:
    random_metrics = run_random_policy_baseline(episodes=10, max_steps=80, seed=42)
    plot_training_metrics(random_metrics, title="Random Policy Baseline - Hospital Environment")


## Lets Run: Optional MAPPO Training
This follows the template run cell, but is disabled by default. Turn it on only after PyTorch works and you want actual training.


In [ ]:
# Template-style training run. Keep False until PyTorch works and you are ready to train.
RUN_MAPPO_TRAINING = False

if RUN_MAPPO_TRAINING:
    actors, critic, training_metrics = train_mappo(
        episodes=250,
        ppo_epochs=5,
        eps_clip=0.2,
        max_steps=100,
        print_every=25,
    )
    plot_training_metrics(training_metrics, title="MAPPO Training - Hospital Environment")
    save_metrics(training_metrics)
